# Baking Lyrics: train the lyrics LSTM

This notebook retrains the Baking Lyrics generator: a conditional word level LSTM language model (AWD-LSTM family) trained in PyTorch, exported to ONNX and served with onnxruntime.

It is a thin driver around the `ml` package of the repository; every step is also a plain command (see `ml/README.md`).

Works on Google Colab (Runtime > Change runtime type > T4, L4 or A100 GPU) and on a local CUDA machine (start Jupyter from the repo with `uv run --group train --with jupyter jupyter lab`).

Steps: 1 setup, 2 data, 3 training, 4 evaluation, 5 ONNX export, 6 try it, 7 publish (on your own machine).

## 1. Setup

Edit the settings below, then run the cells in order. In Colab, training state lives in Google Drive, so a disconnected session resumes where it stopped.

In [ ]:
# Settings
REPO_URL = "https://github.com/arukavina/baking-lyrics.git"
BRANCH = "master"
PRESET = "colab-t4"        # "colab-t4" (T4, 16 GB), "gpu-large" (L4 / A100 / 24 GB+), "smoke" (pipeline check, minutes)
VERSION = "2026.11.0"      # bundle version, used in S3 as models/lyrics-lstm/<VERSION>/
USE_GLOVE = True           # initialise the 300 d embeddings from GloVe 6B (822 MB download, once)
DATA_SOURCE = "kaggle"     # "kaggle" (download the raw MetroLyrics dataset) or "csv" (a lyrics.csv you uploaded)
LOCAL_CSV = ""             # used when DATA_SOURCE == "csv", e.g. "/content/drive/MyDrive/baking-lyrics/lyrics.csv"
EXTRA_SONGDATA = ""        # optional "55000+ Song Lyrics" songdata.csv: MetroLyrics only has artists A to G and "the"
MAX_ARTISTS = 1500         # seed the API with the same value (refresh_database --max-artists)
MIN_SONGS = 20
MAX_SONGS_PER_ARTIST = 150

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not Path("/content/baking-lyrics").exists():
        subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, "/content/baking-lyrics"], check=True)
    os.chdir("/content/baking-lyrics")
    # Colab already ships a CUDA build of torch; install only what is missing.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas", "pyarrow", "kagglehub", "fast-langdetect",
                    "onnx", "onnxscript", "onnxruntime", "tqdm"], check=True)
    from google.colab import drive

    drive.mount("/content/drive")
    WORK = Path("/content/drive/MyDrive/baking-lyrics")
else:
    # Local: run Jupyter from the repository root inside `uv run --group train --with jupyter jupyter lab`.
    while not Path("ml/train.py").exists() and Path.cwd() != Path.cwd().parent:
        os.chdir("..")
    WORK = Path("ml").resolve()

DATA = WORK / "data" / "processed"
RUNS = WORK / "runs"
RUN = RUNS / PRESET
BUNDLES = WORK / "bundles"
for p in (DATA, RUNS, BUNDLES):
    p.mkdir(parents=True, exist_ok=True)
PY = sys.executable


def run(*args):
    # Run `python -m ...`, stream its output into the notebook and raise if it fails.
    env = {**os.environ, "PYTHONUNBUFFERED": "1", "PYTHONUTF8": "1"}
    proc = subprocess.Popen([PY, *map(str, args)], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                            encoding="utf-8", errors="replace", env=env)
    for line in proc.stdout:
        print(line, end="")
    if proc.wait():
        raise RuntimeError(f"command failed with exit code {proc.returncode}")


print("repo:", Path.cwd(), "| work dir:", WORK)

In [ ]:
import torch

print("torch", torch.__version__, "| CUDA", torch.version.cuda)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB",
          "| bf16:", torch.cuda.is_bf16_supported())
else:
    print("No CUDA GPU: only the smoke preset is practical here.")
assert tuple(int(x) for x in torch.__version__.split(".")[:2]) >= (2, 9), "torch >= 2.9 is required for the ONNX export"

## 2. Data

The raw Kaggle dataset `gyani95/380000-lyrics-from-metrolyrics` (version 2) keeps line breaks and punctuation, which the model learns as `<line>` and `<verse>` tokens. Prefer it over the flattened `*_cleaned_lyrics.csv` derivatives.

Kaggle credentials, any one of:
- Colab: add a secret named `KAGGLE_API_TOKEN` (Kaggle > Settings > API > Generate New Token) and give this notebook access;
- environment variable `KAGGLE_API_TOKEN`, or the legacy `KAGGLE_USERNAME` and `KAGGLE_KEY`;
- a legacy `kaggle.json` in `~/.kaggle/`.

Preparation cleans the lyrics (English only with the fastText language id model, no placeholders or instrumentals, deduplicated), keeps the top artists, caps songs per artist and splits train, validation and test by song. It writes `songs.parquet`, `catalog.csv` (for the API catalog seeding), `artists.json` and `prep_meta.json`. It takes about 3 to 5 minutes.

In [ ]:
if IN_COLAB and not os.environ.get("KAGGLE_API_TOKEN"):
    try:
        from google.colab import userdata

        os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
    except Exception as exc:  # secret missing or access not granted
        print("No KAGGLE_API_TOKEN secret:", exc)

In [ ]:
source = ["--kaggle", "--download-dir", str(WORK / "kaggle")] if DATA_SOURCE == "kaggle" else ["--csv", LOCAL_CSV]
if EXTRA_SONGDATA:
    source += ["--extra-songdata", EXTRA_SONGDATA]
if (DATA / "songs.parquet").exists():
    print("Prepared data found in", DATA, "(delete the folder to prepare again)")
else:
    run("-m", "ml.data", *source, "--out", DATA, "--max-artists", MAX_ARTISTS, "--min-songs", MIN_SONGS,
        "--max-songs-per-artist", MAX_SONGS_PER_ARTIST)
print(open(DATA / "prep_meta.json").read())

In [ ]:
glove = []
if USE_GLOVE and PRESET != "smoke":
    zip_path = WORK / "glove.6B.zip"
    if not zip_path.exists():
        # Official Stanford download (822 MB). Mirror: https://zenodo.org/records/4925376 (glove.6B.300d.zip).
        subprocess.run(["wget", "-q", "-O", str(zip_path), "https://nlp.stanford.edu/data/glove.6B.zip"], check=True)
    glove = ["--glove", str(zip_path)]

## 3. Training

Run the next cell; it trains with the chosen preset, evaluates validation perplexity after every epoch, writes samples, keeps `best.pt` and stops early when validation stops improving.

If the session disconnects, run the setup cells and this cell again: training resumes from `last.pt` (saved every 15 minutes and after each epoch).

Expected time (full dataset, about 1,500 artists): T4 around 25 to 40 minutes per epoch with `colab-t4`; L4 or A100 2 to 4 times faster. Add `"--set", "epochs=8"` or `"--set", "max_minutes=200"` to fit a session budget.

In [ ]:
extra = []  # e.g. ["--set", "epochs=8", "--set", "batch_size=24"]
run("-m", "ml.train", "--data", DATA, "--out", RUN, "--preset", PRESET, *glove, *extra)

## 4. Evaluation

In [ ]:
import json

run("-m", "ml.train", "--data", DATA, "--out", RUN, "--eval-only", "--split", "test")
for line in open(RUN / "metrics.jsonl"):
    print(json.loads(line))
print(open(RUN / "metrics.json").read())

In [ ]:
from IPython.display import Markdown, display

display(Markdown(open(RUN / "samples.md", encoding="utf-8").read()[-6000:]))

## 5. ONNX export and parity check

Exports a single step decoder with the recurrent state as explicit input and output, checks onnxruntime against torch (max absolute logit difference below 1e-4) and writes the bundle with samples generated from the ONNX model.

In [ ]:
run("-m", "ml.export", "--run", RUN, "--out", BUNDLES, "--version", VERSION)
BUNDLE = BUNDLES / VERSION
for p in sorted(BUNDLE.iterdir()):
    print(f"{p.name:14} {p.stat().st_size / 1e6:8.1f} MB")
display(Markdown(open(BUNDLE / "samples.md", encoding="utf-8").read()))

## 6. Try it (onnxruntime only, as the API does)

In [ ]:
from ml.generate import LyricsGenerator

gen = LyricsGenerator.from_bundle(BUNDLE)
for artist in gen.artist_slugs[:3]:
    res = gen.generate(artist, seed_text="", number_words=100, seed=42)
    print(f"## {res.title} ({artist})\n{res.lyrics}\n")
print(gen.generate_title("I keep on dancing through the night\nwith you", artist=gen.artist_slugs[0], seed=1))

## 7. Publish (on your machine, with AWS credentials)

The bundle is in Google Drive under `MyDrive/baking-lyrics/bundles/<VERSION>/` (local runs: `ml/bundles/<VERSION>/`).

Download that folder into the repository as `ml\bundles\<VERSION>\`, then in PowerShell from the repository root:

```powershell
uv sync --group train
uv run --group train python -m ml.publish ml\bundles\2026.11.0 --dry-run
uv run --group train python -m ml.publish ml\bundles\2026.11.0 --profile <your-aws-profile>
```

This uploads to `s3://bakinglyrics/models/lyrics-lstm/2026.11.0/` and then switches `models/lyrics-lstm/latest.json`.

Seed the API catalog with the same artists the model knows: `python -m baking.refresh_database --data <DATA>\catalog.csv` (the file is next to `songs.parquet`).